# Network-MCCFR Integration Unit Tests

**Step 5: Minimal Integration Tests**

Tests the connection between DeepCFR network and MCCFR:
- Network predictions on game states
- Regret matching with network output
- Action selection using network
- Comparison with tabular MCCFR

In [ ]:
import sys
import os

# Add parent directory to path
current_dir = os.getcwd()
if current_dir.endswith('deep_CFR_vNB_integration'):
    parent_dir = os.path.dirname(current_dir)
else:
    parent_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.insert(0, parent_dir)

import torch
import random
import numpy as np
from DeepCFR import DeepCFRModule
from mccfr import MCCFR
from integration import NetworkMCCFRIntegration

# Test tracking
tests_passed = 0
tests_failed = 0

def run_test(test_name, test_func):
    global tests_passed, tests_failed
    try:
        test_func()
        print(f'✓ {test_name} PASSED')
        tests_passed += 1
    except AssertionError as e:
        print(f'✗ {test_name} FAILED: {e}')
        tests_failed += 1
    except Exception as e:
        print(f'✗ {test_name} ERROR: {e}')
        tests_failed += 1

print('=' * 70)
print('NETWORK-MCCFR INTEGRATION TESTS')
print('=' * 70)

## 1. Integration Setup Tests

In [ ]:
def test_integration_creates():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    assert integration.network is not None
    assert integration.mccfr is not None

run_test('Integration creates successfully', test_integration_creates)

In [ ]:
def test_network_in_eval_mode():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    assert not integration.network.training, "Network should be in eval mode"

run_test('Network set to eval mode', test_network_in_eval_mode)

## 2. Network Regret Prediction Tests

In [ ]:
def test_get_network_regrets_returns_dict():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    regrets = integration.get_network_regrets(state, player=0)
    assert isinstance(regrets, dict)
    assert len(regrets) > 0

run_test('Network regrets returns dict', test_get_network_regrets_returns_dict)

In [ ]:
def test_network_regrets_are_finite():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    regrets = integration.get_network_regrets(state, player=0)
    for action_key, regret in regrets.items():
        assert np.isfinite(regret), f"Regret for {action_key} is not finite: {regret}"

run_test('Network regrets are finite', test_network_regrets_are_finite)

In [ ]:
def test_network_regrets_only_legal_actions():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    regrets = integration.get_network_regrets(state, player=0)
    legal_actions = mccfr.get_legal_actions_list(state)
    legal_keys = {mccfr.action_to_key(a, state, 0) for a in legal_actions}
    for action_key in regrets.keys():
        assert action_key in legal_keys, f"{action_key} not in legal actions"

run_test('Network regrets only for legal actions', test_network_regrets_only_legal_actions)

## 3. Network Strategy Tests

In [ ]:
def test_network_strategy_sums_to_one():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    strategy = integration.get_network_strategy(state, player=0)
    total_prob = sum(strategy.values())
    assert abs(total_prob - 1.0) < 0.001, f"Strategy should sum to 1.0, got {total_prob}"

run_test('Network strategy sums to 1.0', test_network_strategy_sums_to_one)

In [ ]:
def test_network_strategy_all_non_negative():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    strategy = integration.get_network_strategy(state, player=0)
    for action_key, prob in strategy.items():
        assert prob >= 0.0, f"Probability for {action_key} is negative: {prob}"

run_test('Network strategy all non-negative', test_network_strategy_all_non_negative)

In [ ]:
def test_network_strategy_deterministic():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    strategy1 = integration.get_network_strategy(state, player=0)
    strategy2 = integration.get_network_strategy(state, player=0)
    for action_key in strategy1.keys():
        assert abs(strategy1[action_key] - strategy2[action_key]) < 0.001

run_test('Network strategy is deterministic', test_network_strategy_deterministic)

## 4. Action Selection Tests

In [ ]:
def test_select_action_returns_legal_action():
    random.seed(42)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    legal_types = {type(a).__name__ for a in legal_actions}
    for _ in range(10):
        action = integration.select_network_action(state, player=0)
        assert type(action).__name__ in legal_types

run_test('Select action returns legal action', test_select_action_returns_legal_action)

In [ ]:
def test_select_action_samples_distribution():
    random.seed(123)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    action_counts = {}
    n_samples = 100
    for _ in range(n_samples):
        action = integration.select_network_action(state, player=0)
        action_type = type(action).__name__
        action_counts[action_type] = action_counts.get(action_type, 0) + 1
    assert len(action_counts) > 0, "Should sample at least one action type"
    print(f"  Sampled {len(action_counts)} different action types from {n_samples} samples")

run_test('Select action samples from distribution', test_select_action_samples_distribution)

## 5. Network vs Tabular Comparison Tests

In [ ]:
def test_comparison_returns_complete_data():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    comparison = integration.compare_network_vs_tabular(state, player=0)
    assert 'infoset' in comparison
    assert 'network_regrets' in comparison
    assert 'tabular_regrets' in comparison
    assert 'network_strategy' in comparison
    assert 'tabular_strategy' in comparison
    assert 'legal_actions' in comparison

run_test('Comparison returns complete data', test_comparison_returns_complete_data)

In [ ]:
def test_comparison_with_trained_mccfr():
    random.seed(456)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    
    # Train MCCFR for a few iterations
    for i in range(5):
        state = mccfr.create_initial_state()
        mccfr.external_sampling(state, i % 2)
    
    # Now compare
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    comparison = integration.compare_network_vs_tabular(state, player=0)
    
    # Both strategies should exist
    assert len(comparison['network_strategy']) > 0
    assert len(comparison['tabular_strategy']) > 0
    
    # Both should sum to 1
    net_sum = sum(comparison['network_strategy'].values())
    tab_sum = sum(comparison['tabular_strategy'].values())
    assert abs(net_sum - 1.0) < 0.001
    assert abs(tab_sum - 1.0) < 0.001

run_test('Comparison with trained MCCFR', test_comparison_with_trained_mccfr)

## 6. Multi-State Tests

In [ ]:
def test_integration_on_multiple_states():
    random.seed(789)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    
    # Test on 10 different states
    for i in range(10):
        state = mccfr.create_initial_state()
        regrets = integration.get_network_regrets(state, player=0)
        strategy = integration.get_network_strategy(state, player=0)
        action = integration.select_network_action(state, player=0)
        
        assert len(regrets) > 0
        assert abs(sum(strategy.values()) - 1.0) < 0.001
        assert action is not None
    
    print(f"  Successfully tested integration on 10 different states")

run_test('Integration on multiple states', test_integration_on_multiple_states)

In [ ]:
def test_integration_both_players():
    random.seed(101)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    state = mccfr.create_initial_state()
    
    # Test for both players
    for player in [0, 1]:
        regrets = integration.get_network_regrets(state, player=player)
        strategy = integration.get_network_strategy(state, player=player)
        assert len(regrets) > 0
        assert abs(sum(strategy.values()) - 1.0) < 0.001
    
    print(f"  Successfully tested integration for both players")

run_test('Integration for both players', test_integration_both_players)

## 7. Edge Case Tests

In [ ]:
def test_integration_with_single_legal_action():
    """Test when only one action is legal (edge case)"""
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    integration = NetworkMCCFRIntegration(network, mccfr)
    
    # Create state and check if it has single action (rare)
    state = mccfr.create_initial_state()
    legal_actions = mccfr.get_legal_actions_list(state)
    
    # At preflop, there are usually multiple actions, but test the code path
    if len(legal_actions) == 1:
        strategy = integration.get_network_strategy(state, player=0)
        # Single action should get probability 1.0
        assert abs(list(strategy.values())[0] - 1.0) < 0.001
    
    print(f"  Tested with {len(legal_actions)} legal actions")

run_test('Integration with edge cases', test_integration_with_single_legal_action)

## Test Summary

In [ ]:
print('\n' + '=' * 70)
print('NETWORK-MCCFR INTEGRATION TEST SUMMARY')
print('=' * 70)
print(f'\nTests passed: {tests_passed}')
print(f'Tests failed: {tests_failed}')
print(f'Total tests: {tests_passed + tests_failed}')
if tests_failed == 0:
    print('\n✓✓✓ ALL INTEGRATION TESTS PASSED! ✓✓✓')
    print('\nIntegration verified for:')
    print('  ✓ Network regret predictions')
    print('  ✓ Strategy computation (regret matching)')
    print('  ✓ Action selection')
    print('  ✓ Network vs tabular comparison')
    print('  ✓ Multiple states and both players')
    print('\n🎉 Ready to proceed to Step 6: Training Pipeline')
else:
    print(f'\n✗ {tests_failed} TEST(S) FAILED')
print('=' * 70)